<a href="https://colab.research.google.com/github/eunmindow-star/chaeeunmin/blob/main/code/10_table1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

# 전체 표본(54,170명)을 유지한 상태에서 도메인 내 비율/평균 및 설계 기반 SE를 계산하는 함수
def calc_survey_domain_stats(df, domain_cond, target_var, is_binary=True, strata_col='STRATA', cluster_col='CLUSTER', weight_col='W'):
    """
    전체 표본 설계 구조(STRATA, CLUSTER, W)를 유지하면서 특정 도메인(domain_cond)의
    가중 평균(또는 비율)과 설계 기반 표준오차(Taylor Linearization SE)를 산출합니다.
    """
    # 도메인 내 유효 응답자 조건 (도메인에 속하고, target_var가 결측이 아님)
    in_domain = domain_cond & df[target_var].notna()

    # 비가중 관측 수 (n)
    n_unweighted = in_domain.sum()

    # 도메인 지시변수 (도메인 내 유효 응답자면 1, 아니면 0)
    d = in_domain.astype(float).values
    y = np.where(in_domain, df[target_var].fillna(0).values, 0.0)
    w = df[weight_col].values

    # 가중 합계 및 점추정치 (Ratio Estimator: Y_hat / D_hat)
    w_d = np.sum(w * d)
    w_y = np.sum(w * y)
    est = w_y / w_d  # 가중 평균 (이진변수일 경우 가중 비율)

    # Taylor 선형화 잔차 (Linearized residual): z_i = (d_i * (y_i - est)) / w_d
    z = (d * (y - est)) / w_d
    wz = w * z

    # 층화(STRATA) 및 집락(CLUSTER) 기반 분산 추정 (With-replacement approximation)
    temp_df = pd.DataFrame({
        'strata': df[strata_col].values,
        'cluster': df[cluster_col].values,
        'wz': wz
    })

    # 집락(PSU) 단위 합계 계산
    psu_sums = temp_df.groupby(['strata', 'cluster'])['wz'].sum().reset_index()

    # 층별 집락 수(n_h)와 층별 평균 계산
    strata_stats = psu_sums.groupby('strata')['wz'].agg(['count', 'mean']).rename(columns={'count': 'n_h', 'mean': 'mean_h'})
    psu_sums = psu_sums.merge(strata_stats, on='strata')

    # 층별 분산 합산: sum_h [ (n_h / (n_h - 1)) * sum_i (wz_hi - mean_h)^2 ]
    psu_sums['sq_diff'] = (psu_sums['wz'] - psu_sums['mean_h']) ** 2
    strata_var = psu_sums.groupby('strata').apply(
        lambda g: (g['n_h'].iloc[0] / (g['n_h'].iloc[0] - 1)) * g['sq_diff'].sum() if g['n_h'].iloc[0] > 1 else 0
    )

    total_var = strata_var.sum()
    se = np.sqrt(total_var)

    if is_binary:
        return n_unweighted, est * 100, se * 100  # 비율(%)과 SE(%p) 반환
    else:
        return n_unweighted, est, se              # 평균과 SE 반환

In [6]:
import pandas as pd
import numpy as np

# 1. 실제 SAS 데이터 불러오기
file_path = '/content/drive/MyDrive/hdac_data/kyrbs2025.sas7bdat'
df = pd.read_sas(file_path)
print("데이터 로드 성공! 전체 행 수:", len(df))

# 2. 비만/BMI 관련 공식 변수(OB 등)가 있는지 확인하기
ob_cols = [c for c in df.columns if 'OB' in c.upper() or 'BMI' in c.upper()]
print("비만/BMI 관련 변수:", ob_cols)

if 'OB' in df.columns:
    print("\n=== OB 변수 빈도표 ===")
    print(df['OB'].value_counts(dropna=False))

데이터 로드 성공! 전체 행 수: 54170
비만/BMI 관련 변수: ['OBS', 'E_FM_OBS_7']


In [7]:
# OBS 변수 코드 분포 확인 (고등학생 기준)
print("=== OBS 전체 빈도표 ===")
print(df['OBS'].value_counts(dropna=False))

# 고등학생(GRADE 4, 5, 6)에서 OBS 코드별 평균 BMI 확인 (어느 코드가 비만인지 100% 확실히 검증)
df['bmi_temp'] = df['WT'] / ((df['HT'] / 100) ** 2)
print("\n=== OBS 코드별 평균 BMI ===")
print(df[df['GRADE'].isin([4, 5, 6])].groupby('OBS')['bmi_temp'].agg(['count', 'mean', 'min', 'max']))

=== OBS 전체 빈도표 ===
OBS
b'A100001'    1
b'A100002'    1
b'A100003'    1
b'A100004'    1
b'A100005'    1
             ..
b'A154166'    1
b'A154167'    1
b'A154168'    1
b'A154169'    1
b'A154170'    1
Name: count, Length: 54170, dtype: int64

=== OBS 코드별 평균 BMI ===
            count       mean        min        max
OBS                                               
b'A103767'      1  23.645882  23.645882  23.645882
b'A103768'      1  23.388687  23.388687  23.388687
b'A103769'      1  22.667997  22.667997  22.667997
b'A103770'      1  20.715694  20.715694  20.715694
b'A103771'      1  23.292876  23.292876  23.292876
...           ...        ...        ...        ...
b'A154166'      1  31.583112  31.583112  31.583112
b'A154167'      0        NaN        NaN        NaN
b'A154168'      1  23.378796  23.378796  23.378796
b'A154169'      1  31.141869  31.141869  31.141869
b'A154170'      1  23.120624  23.120624  23.120624

[25843 rows x 4 columns]


/tmp/ipykernel_694/3776970191.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['bmi_temp'] = df['WT'] / ((df['HT'] / 100) ** 2)


In [8]:
import numpy as np
import pandas as pd

# ==========================================
# 0. 기본 변수 생성 (전체 54,170명 표본설계 구조 유지)
# ==========================================
df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)
df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
                         np.where(df['F_BR'].notnull(), 0, np.nan))
df['bmi'] = df['WT'] / ((df['HT'] / 100) ** 2)
df['obesity'] = np.where(df['bmi'] >= 25, 1,
                         np.where(df['bmi'].notnull(), 0, np.nan))

# 최종 분석 유효 대상자 표식
df['valid_sample'] = (df['domain_hs'] == 1) & df['bf_skip'].notnull() & df['bmi'].notnull()

# ==========================================
# 1. [B-1] 제외 인원 766명 세부 사유 검증 출력
# ==========================================
print("==========================================")
print(" [B-1] 분석 대상자 선정 및 제외 인원 세부 검증")
print("==========================================")
hs = df[df['GRADE'].isin([4, 5, 6])].copy()
miss_fbr = hs['F_BR'].isna()
miss_ht = hs['HT'].isna()
miss_wt = hs['WT'].isna()

only_ht = miss_ht & ~miss_wt & ~miss_fbr
only_wt = ~miss_ht & miss_wt & ~miss_fbr
both_hw = miss_ht & miss_wt & ~miss_fbr
any_miss = miss_fbr | miss_ht | miss_wt

print(f"1단계 전체 조사 표본: {len(df):,}명")
print(f"2단계 고등학생 도메인 추출(고1~고3): {len(hs):,}명 (중학생 제외: {len(df)-len(hs):,}명)")
print(f"3단계 결측 제외 총 인원: {any_miss.sum():,}명")
print(f"   - 아침식사(F_BR) 무응답: {miss_fbr.sum():,}명")
print(f"   - 신장(HT)만 결측: {only_ht.sum():,}명")
print(f"   - 체중(WT)만 결측: {only_wt.sum():,}명")
print(f"   - 신장(HT)·체중(WT) 동시 결측: {both_hw.sum():,}명")
print(f"4단계 최종 유효 분석 대상자: {df['valid_sample'].sum():,}명")
print(f"   - 결식군(주 5회 이상 결식): {(df['valid_sample'] & (df['bf_skip']==1)).sum():,}명")
print(f"   - 비결식군(주 0~4회 결식): {(df['valid_sample'] & (df['bf_skip']==0)).sum():,}명\n")

# ==========================================
# 2. [B-2] 설계 기반 SE 포함 Table 1 자동 출력
# ==========================================
cond_total = (df['valid_sample'] == True)
cond_skip  = (df['valid_sample'] == True) & (df['bf_skip'] == 1)
cond_noskip = (df['valid_sample'] == True) & (df['bf_skip'] == 0)

# 범주형 더미변수 생성
df['sex_m'] = np.where(df['SEX'] == 1, 1, np.where(df['SEX'].notna(), 0, np.nan))
df['sex_f'] = np.where(df['SEX'] == 2, 1, np.where(df['SEX'].notna(), 0, np.nan))
df['g1'] = np.where(df['GRADE'] == 4, 1, 0)
df['g2'] = np.where(df['GRADE'] == 5, 1, 0)
df['g3'] = np.where(df['GRADE'] == 6, 1, 0)
df['ob_0'] = np.where(df['obesity'] == 0, 1, np.where(df['obesity'].notna(), 0, np.nan))
df['ob_1'] = np.where(df['obesity'] == 1, 1, np.where(df['obesity'].notna(), 0, np.nan))

variables = [
    ('성별 - 남학생 (Male)', 'sex_m', 'SEX', True),
    ('성별 - 여학생 (Female)', 'sex_f', 'SEX', True),
    ('학년 - 고등학교 1학년', 'g1', 'GRADE', True),
    ('학년 - 고등학교 2학년', 'g2', 'GRADE', True),
    ('학년 - 고등학교 3학년', 'g3', 'GRADE', True),
    ('비만 여부 - 비비만 (BMI < 25)', 'ob_0', 'obesity', True),
    ('비만 여부 - 비만 (BMI ≥ 25)', 'ob_1', 'obesity', True),
    ('체질량지수 (BMI, kg/m²)', 'bmi', 'bmi', False),
]

t1_rows = []
for label, col, orig_col, is_bin in variables:
    # 유효 대상자(25,077명) 내 해당 변수의 결측 수 (고등학생 전체 기준 결측 수도 함께 확인 가능)
    miss_valid = df.loc[cond_total, orig_col].isna().sum()
    miss_hs = hs[orig_col].isna().sum()

    n_t, est_t, se_t = calc_survey_domain_stats(df, cond_total, col, is_bin)
    n_s, est_s, se_s = calc_survey_domain_stats(df, cond_skip, col, is_bin)
    n_ns, est_ns, se_ns = calc_survey_domain_stats(df, cond_noskip, col, is_bin)

    if is_bin:
        fmt = lambda n, e, s: f"{n:,} ({e:.2f}% [SE {s:.2f}])"
    else:
        fmt = lambda n, e, s: f"{e:.2f} (SE {s:.4f})"

    t1_rows.append({
        '특성 (Characteristic)': label,
        '결측 수 (유효표본/고등전체)': f"{miss_valid} / {miss_hs}",
        f'전체 (N={cond_total.sum():,})': fmt(n_t, est_t, se_t),
        f'결식군 (N={cond_skip.sum():,})': fmt(n_s, est_s, se_s),
        f'비결식군 (N={cond_noskip.sum():,})': fmt(n_ns, est_ns, se_ns)
    })

print("==========================================")
print(" [B-2] Table 1 (복합표본 설계 기반 가중치 및 SE 반영)")
print("==========================================")
display(pd.DataFrame(t1_rows))

# ==========================================
# 3. [B-3] 비가중 vs 가중 전체 비교표 자동 출력
# ==========================================
b3_rows = []
compare_vars = [('아침식사 결식률 (주 5회 이상)', 'bf_skip', True)] + [(v[0], v[1], v[3]) for v in variables]

for label, col, is_bin in compare_vars:
    n_w, est_w, se_w = calc_survey_domain_stats(df, cond_total, col, is_bin)
    sub_vals = df.loc[cond_total, col].dropna()

    if is_bin:
        est_unw = (sub_vals.sum() / len(sub_vals)) * 100
        diff = est_w - est_unw
        b3_rows.append({
            '변수명': label,
            '비가중 추정치': f"{int(sub_vals.sum()):,} ({est_unw:.2f}%)",
            '가중 추정치': f"{est_w:.2f}% (SE {se_w:.2f})",
            '차이 (가중 - 비가중)': f"{diff:+.2f}%p"
        })
    else:
        est_unw = sub_vals.mean()
        se_unw = sub_vals.std(ddof=1) / np.sqrt(len(sub_vals))
        diff = est_w - est_unw
        b3_rows.append({
            '변수명': label,
            '비가중 추정치': f"{est_unw:.2f} (SE {se_unw:.4f})",
            '가중 추정치': f"{est_w:.2f} (SE {se_w:.4f})",
            '차이 (가중 - 비가중)': f"{diff:+.2f}"
        })

print("\n==========================================")
print(" [B-3] 비가중 vs 가중 전체 비교표 및 차이 자동 산출")
print("==========================================")
display(pd.DataFrame(b3_rows))

/tmp/ipykernel_694/2433222127.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)
/tmp/ipykernel_694/2433222127.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
/tmp/ipykernel_694/2433222127.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To

 [B-1] 분석 대상자 선정 및 제외 인원 세부 검증
1단계 전체 조사 표본: 54,170명
2단계 고등학생 도메인 추출(고1~고3): 25,843명 (중학생 제외: 28,327명)
3단계 결측 제외 총 인원: 766명
   - 아침식사(F_BR) 무응답: 0명
   - 신장(HT)만 결측: 0명
   - 체중(WT)만 결측: 0명
   - 신장(HT)·체중(WT) 동시 결측: 766명
4단계 최종 유효 분석 대상자: 25,077명
   - 결식군(주 5회 이상 결식): 11,376명
   - 비결식군(주 0~4회 결식): 13,701명



/tmp/ipykernel_694/2433222127.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['sex_m'] = np.where(df['SEX'] == 1, 1, np.where(df['SEX'].notna(), 0, np.nan))
/tmp/ipykernel_694/2433222127.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['sex_f'] = np.where(df['SEX'] == 2, 1, np.where(df['SEX'].notna(), 0, np.nan))
/tmp/ipykernel_694/2433222127.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all

 [B-2] Table 1 (복합표본 설계 기반 가중치 및 SE 반영)


/tmp/ipykernel_694/274047026.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  strata_var = psu_sums.groupby('strata').apply(
/tmp/ipykernel_694/274047026.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  strata_var = psu_sums.groupby('strata').apply(
/tmp/ipykernel_694/274047026.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a f

,특성 (Characteristic),결측 수 (유효표본/고등전체),"전체 (N=25,077)","결식군 (N=11,376)","비결식군 (N=13,701)"
0,성별 - 남학생 (Male),0 / 0,"25,077 (51.55% [SE 1.93])","11,376 (50.02% [SE 1.95])","13,701 (52.84% [SE 1.99])"
1,성별 - 여학생 (Female),0 / 0,"25,077 (48.45% [SE 1.93])","11,376 (49.98% [SE 1.95])","13,701 (47.16% [SE 1.99])"
2,학년 - 고등학교 1학년,0 / 0,"25,077 (32.83% [SE 0.21])","11,376 (31.23% [SE 0.40])","13,701 (34.18% [SE 0.36])"
3,학년 - 고등학교 2학년,0 / 0,"25,077 (33.17% [SE 0.19])","11,376 (34.13% [SE 0.42])","13,701 (32.36% [SE 0.36])"
4,학년 - 고등학교 3학년,0 / 0,"25,077 (34.00% [SE 0.22])","11,376 (34.64% [SE 0.45])","13,701 (33.46% [SE 0.40])"
5,비만 여부 - 비비만 (BMI < 25),0 / 766,"25,077 (79.94% [SE 0.46])","11,376 (79.35% [SE 0.55])","13,701 (80.44% [SE 0.53])"
6,비만 여부 - 비만 (BMI ≥ 25),0 / 766,"25,077 (20.06% [SE 0.46])","11,376 (20.65% [SE 0.55])","13,701 (19.56% [SE 0.53])"
7,"체질량지수 (BMI, kg/m²)",0 / 766,22.18 (SE 0.0521),22.27 (SE 0.0569),22.11 (SE 0.0570)


/tmp/ipykernel_694/274047026.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  strata_var = psu_sums.groupby('strata').apply(
/tmp/ipykernel_694/274047026.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  strata_var = psu_sums.groupby('strata').apply(
/tmp/ipykernel_694/274047026.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a f


 [B-3] 비가중 vs 가중 전체 비교표 및 차이 자동 산출


/tmp/ipykernel_694/274047026.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  strata_var = psu_sums.groupby('strata').apply(
/tmp/ipykernel_694/274047026.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  strata_var = psu_sums.groupby('strata').apply(
/tmp/ipykernel_694/274047026.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a f

,변수명,비가중 추정치,가중 추정치,차이 (가중 - 비가중)
0,아침식사 결식률 (주 5회 이상),"11,376 (45.36%)",45.68% (SE 0.49),+0.32%p
1,성별 - 남학생 (Male),"12,949 (51.64%)",51.55% (SE 1.93),-0.09%p
2,성별 - 여학생 (Female),"12,128 (48.36%)",48.45% (SE 1.93),+0.09%p
3,학년 - 고등학교 1학년,"8,526 (34.00%)",32.83% (SE 0.21),-1.17%p
4,학년 - 고등학교 2학년,"8,363 (33.35%)",33.17% (SE 0.19),-0.18%p
5,학년 - 고등학교 3학년,"8,188 (32.65%)",34.00% (SE 0.22),+1.35%p
6,비만 여부 - 비비만 (BMI < 25),"19,933 (79.49%)",79.94% (SE 0.46),+0.46%p
7,비만 여부 - 비만 (BMI ≥ 25),"5,144 (20.51%)",20.06% (SE 0.46),-0.46%p
8,"체질량지수 (BMI, kg/m²)",22.23 (SE 0.0240),22.18 (SE 0.0521),-0.04


In [9]:
import warnings
warnings.filterwarnings('ignore')  # 불필요한 판다스 경고문 숨기기

from google.colab import drive
import pandas as pd
import numpy as np

# 1. 구글 드라이브 연결 및 원시자료 불러오기
drive.mount('/content/drive')
df = pd.read_sas('/content/drive/MyDrive/hdac_data/kyrbs2025.sas7bdat')

# 2. 파생변수 생성 (전체 54,170명 표본설계 구조 유지)
new_cols = pd.DataFrame({
    'domain_hs': df['GRADE'].isin([4, 5, 6]).astype(int),
    'bf_skip': np.where(df['F_BR'].isin([1, 2, 3]), 1, np.where(df['F_BR'].notnull(), 0, np.nan)),
    'bmi': df['WT'] / ((df['HT'] / 100) ** 2),
}, index=df.index)
new_cols['obesity'] = np.where(new_cols['bmi'] >= 25, 1, np.where(new_cols['bmi'].notnull(), 0, np.nan))
new_cols['valid_sample'] = (new_cols['domain_hs'] == 1) & new_cols['bf_skip'].notnull() & new_cols['bmi'].notnull()

# 더미변수 생성
new_cols['sex_m'] = np.where(df['SEX'] == 1, 1, np.where(df['SEX'].notna(), 0, np.nan))
new_cols['sex_f'] = np.where(df['SEX'] == 2, 1, np.where(df['SEX'].notna(), 0, np.nan))
new_cols['g1'] = np.where(df['GRADE'] == 4, 1, 0)
new_cols['g2'] = np.where(df['GRADE'] == 5, 1, 0)
new_cols['g3'] = np.where(df['GRADE'] == 6, 1, 0)
new_cols['ob_0'] = np.where(new_cols['obesity'] == 0, 1, np.where(new_cols['obesity'].notna(), 0, np.nan))
new_cols['ob_1'] = np.where(new_cols['obesity'] == 1, 1, np.where(new_cols['obesity'].notna(), 0, np.nan))

df = pd.concat([df, new_cols], axis=1)

# 3. 복합표본 설계 기반 도메인 통계량 및 SE 산출 함수 (W, STRATA, CLUSTER 반영)
def calc_survey_domain_stats(data, domain_cond, target_var, is_binary=True, strata_col='STRATA', cluster_col='CLUSTER', weight_col='W'):
    in_domain = domain_cond & data[target_var].notna()

    # 비가중 관측 수(n): 이진변수면 해당 범주(1)의 인원수, 연속형이면 유효 응답자 수
    if is_binary:
        n_unweighted = int((in_domain & (data[target_var] == 1)).sum())
    else:
        n_unweighted = int(in_domain.sum())

    d = in_domain.astype(float).values
    y = np.where(in_domain, data[target_var].fillna(0).values, 0.0)
    w = data[weight_col].values

    w_d = np.sum(w * d)
    w_y = np.sum(w * y)
    est = w_y / w_d

    # Taylor 선형화 잔차
    z = (d * (y - est)) / w_d
    wz = w * z

    temp_df = pd.DataFrame({'strata': data[strata_col].values, 'cluster': data[cluster_col].values, 'wz': wz})
    psu_sums = temp_df.groupby(['strata', 'cluster'])['wz'].sum().reset_index()
    strata_stats = psu_sums.groupby('strata')['wz'].agg(['count', 'mean']).rename(columns={'count': 'n_h', 'mean': 'mean_h'})
    psu_sums = psu_sums.merge(strata_stats, on='strata')
    psu_sums['sq_diff'] = (psu_sums['wz'] - psu_sums['mean_h']) ** 2

    strata_var = psu_sums.groupby('strata', group_keys=False).apply(
        lambda g: (g['n_h'].iloc[0] / (g['n_h'].iloc[0] - 1)) * g['sq_diff'].sum() if g['n_h'].iloc[0] > 1 else 0
    )
    se = np.sqrt(strata_var.sum())

    return (n_unweighted, est * 100, se * 100) if is_binary else (n_unweighted, est, se)

# 4. [B-1] 분석 대상자 선정 및 제외 인원 세부 검증
print("==========================================")
print(" [B-1] 분석 대상자 선정 및 제외 인원 세부 검증")
print("==========================================")
hs = df[df['GRADE'].isin([4, 5, 6])]
miss_fbr = hs['F_BR'].isna()
miss_ht = hs['HT'].isna()
miss_wt = hs['WT'].isna()

print(f"1단계 전체 조사 표본: {len(df):,}명")
print(f"2단계 고등학생 도메인 추출(고1~고3): {len(hs):,}명 (중학생 제외: {len(df)-len(hs):,}명)")
print(f"3단계 결측 제외 총 인원: {(miss_fbr | miss_ht | miss_wt).sum():,}명")
print(f"   - 아침식사(F_BR) 무응답: {miss_fbr.sum():,}명")
print(f"   - 신장(HT)만 결측: {(miss_ht & ~miss_wt & ~miss_fbr).sum():,}명")
print(f"   - 체중(WT)만 결측: {(~miss_ht & miss_wt & ~miss_fbr).sum():,}명")
print(f"   - 신장(HT)·체중(WT) 동시 결측: {(miss_ht & miss_wt & ~miss_fbr).sum():,}명")
print(f"4단계 최종 유효 분석 대상자: {df['valid_sample'].sum():,}명")
print(f"   - 결식군(주 5회 이상 결식): {(df['valid_sample'] & (df['bf_skip']==1)).sum():,}명")
print(f"   - 비결식군(주 0~4회 결식): {(df['valid_sample'] & (df['bf_skip']==0)).sum():,}명\n")

# 5. [B-2] Table 1 자동 출력
cond_total = (df['valid_sample'] == True)
cond_skip  = (df['valid_sample'] == True) & (df['bf_skip'] == 1)
cond_noskip = (df['valid_sample'] == True) & (df['bf_skip'] == 0)

variables = [
    ('성별 - 남학생 (Male)', 'sex_m', 'SEX', True),
    ('성별 - 여학생 (Female)', 'sex_f', 'SEX', True),
    ('학년 - 고등학교 1학년', 'g1', 'GRADE', True),
    ('학년 - 고등학교 2학년', 'g2', 'GRADE', True),
    ('학년 - 고등학교 3학년', 'g3', 'GRADE', True),
    ('비만 여부 - 비비만 (BMI < 25)', 'ob_0', 'obesity', True),
    ('비만 여부 - 비만 (BMI ≥ 25)', 'ob_1', 'obesity', True),
    ('체질량지수 (BMI, kg/m²)', 'bmi', 'bmi', False),
]

t1_rows = []
for label, col, orig_col, is_bin in variables:
    miss_valid = int(df.loc[cond_total, orig_col].isna().sum())
    miss_hs = int(hs[orig_col].isna().sum())

    n_t, est_t, se_t = calc_survey_domain_stats(df, cond_total, col, is_bin)
    n_s, est_s, se_s = calc_survey_domain_stats(df, cond_skip, col, is_bin)
    n_ns, est_ns, se_ns = calc_survey_domain_stats(df, cond_noskip, col, is_bin)

    fmt = (lambda n, e, s: f"{n:,} ({e:.2f}% [SE {s:.2f}])") if is_bin else (lambda n, e, s: f"{e:.2f} (SE {s:.4f})")
    t1_rows.append({
        '특성 (Characteristic)': label,
        '결측 수 (유효/고등전체)': f"{miss_valid} / {miss_hs}",
        f'전체 (N={cond_total.sum():,})': fmt(n_t, est_t, se_t),
        f'결식군 (N={cond_skip.sum():,})': fmt(n_s, est_s, se_s),
        f'비결식군 (N={cond_noskip.sum():,})': fmt(n_ns, est_ns, se_ns)
    })

print("==========================================")
print(" [B-2] Table 1 (복합표본 설계 기반 가중치 및 SE 반영)")
print("==========================================")
display(pd.DataFrame(t1_rows))

# 6. [B-3] 비가중 vs 가중 전체 비교표 자동 출력
b3_rows = []
compare_vars = [('아침식사 결식률 (주 5회 이상)', 'bf_skip', True)] + [(v[0], v[1], v[3]) for v in variables]

for label, col, is_bin in compare_vars:
    n_w, est_w, se_w = calc_survey_domain_stats(df, cond_total, col, is_bin)
    sub_vals = df.loc[cond_total, col].dropna()

    if is_bin:
        est_unw = (sub_vals.sum() / len(sub_vals)) * 100
        diff = est_w - est_unw
        b3_rows.append({
            '변수명': label,
            '비가중 추정치': f"{int(sub_vals.sum()):,} ({est_unw:.2f}%)",
            '가중 추정치': f"{est_w:.2f}% (SE {se_w:.2f})",
            '차이 (가중 - 비가중)': f"{diff:+.2f}%p"
        })
    else:
        est_unw = sub_vals.mean()
        se_unw = sub_vals.std(ddof=1) / np.sqrt(len(sub_vals))
        diff = est_w - est_unw
        b3_rows.append({
            '변수명': label,
            '비가중 추정치': f"{est_unw:.2f} (SE {se_unw:.4f})",
            '가중 추정치': f"{est_w:.2f} (SE {se_w:.4f})",
            '차이 (가중 - 비가중)': f"{diff:+.2f}"
        })

print("\n==========================================")
print(" [B-3] 비가중 vs 가중 전체 비교표 및 차이 자동 산출")
print("==========================================")
display(pd.DataFrame(b3_rows))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 [B-1] 분석 대상자 선정 및 제외 인원 세부 검증
1단계 전체 조사 표본: 54,170명
2단계 고등학생 도메인 추출(고1~고3): 25,843명 (중학생 제외: 28,327명)
3단계 결측 제외 총 인원: 766명
   - 아침식사(F_BR) 무응답: 0명
   - 신장(HT)만 결측: 0명
   - 체중(WT)만 결측: 0명
   - 신장(HT)·체중(WT) 동시 결측: 766명
4단계 최종 유효 분석 대상자: 25,077명
   - 결식군(주 5회 이상 결식): 11,376명
   - 비결식군(주 0~4회 결식): 13,701명

 [B-2] Table 1 (복합표본 설계 기반 가중치 및 SE 반영)


,특성 (Characteristic),결측 수 (유효/고등전체),"전체 (N=25,077)","결식군 (N=11,376)","비결식군 (N=13,701)"
0,성별 - 남학생 (Male),0 / 0,"12,949 (51.55% [SE 1.93])","5,697 (50.02% [SE 1.95])","7,252 (52.84% [SE 1.99])"
1,성별 - 여학생 (Female),0 / 0,"12,128 (48.45% [SE 1.93])","5,679 (49.98% [SE 1.95])","6,449 (47.16% [SE 1.99])"
2,학년 - 고등학교 1학년,0 / 0,"8,526 (32.83% [SE 0.21])","3,676 (31.23% [SE 0.40])","4,850 (34.18% [SE 0.36])"
3,학년 - 고등학교 2학년,0 / 0,"8,363 (33.17% [SE 0.19])","3,903 (34.13% [SE 0.42])","4,460 (32.36% [SE 0.36])"
4,학년 - 고등학교 3학년,0 / 0,"8,188 (34.00% [SE 0.22])","3,797 (34.64% [SE 0.45])","4,391 (33.46% [SE 0.40])"
5,비만 여부 - 비비만 (BMI < 25),0 / 766,"19,933 (79.94% [SE 0.46])","8,976 (79.35% [SE 0.55])","10,957 (80.44% [SE 0.53])"
6,비만 여부 - 비만 (BMI ≥ 25),0 / 766,"5,144 (20.06% [SE 0.46])","2,400 (20.65% [SE 0.55])","2,744 (19.56% [SE 0.53])"
7,"체질량지수 (BMI, kg/m²)",0 / 766,22.18 (SE 0.0521),22.27 (SE 0.0569),22.11 (SE 0.0570)



 [B-3] 비가중 vs 가중 전체 비교표 및 차이 자동 산출


,변수명,비가중 추정치,가중 추정치,차이 (가중 - 비가중)
0,아침식사 결식률 (주 5회 이상),"11,376 (45.36%)",45.68% (SE 0.49),+0.32%p
1,성별 - 남학생 (Male),"12,949 (51.64%)",51.55% (SE 1.93),-0.09%p
2,성별 - 여학생 (Female),"12,128 (48.36%)",48.45% (SE 1.93),+0.09%p
3,학년 - 고등학교 1학년,"8,526 (34.00%)",32.83% (SE 0.21),-1.17%p
4,학년 - 고등학교 2학년,"8,363 (33.35%)",33.17% (SE 0.19),-0.18%p
5,학년 - 고등학교 3학년,"8,188 (32.65%)",34.00% (SE 0.22),+1.35%p
6,비만 여부 - 비비만 (BMI < 25),"19,933 (79.49%)",79.94% (SE 0.46),+0.46%p
7,비만 여부 - 비만 (BMI ≥ 25),"5,144 (20.51%)",20.06% (SE 0.46),-0.46%p
8,"체질량지수 (BMI, kg/m²)",22.23 (SE 0.0240),22.18 (SE 0.0521),-0.04
